# Download FineWeb Data
Streams the `sample-10BT` subset of `HuggingFaceFW/fineweb` (~10B tokens, ~28 GB raw text) into `data_fineweb/`.
Resumable — detects existing files and skips ahead.

**Next:** run the tokenizer pipeline WITHOUT retraining (`run_tokenizer.py` with no `--train`, or the notebook with `TRAIN_NEW_TOKENIZER=False`), then upload, then pretrain. Retraining the tokenizer would change the tokenization_id and wipe every existing shard.

In [ ]:
!pip install -q datasets

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/synapse/datasets_pretrain/data_fineweb"
os.makedirs(OUT_DIR, exist_ok=True)

# sample-10BT has ~14-15M docs; the cap is just a safety stop — the stream
# ends naturally when the subset is exhausted.
FINEWEB_SUBSET = "sample-10BT"
MAX_DOCS = 20_000_000
DOCS_PER_FILE = 10_000
SEPARATOR = "\n<|endoftext|>\n"

# Check what we already have (for resume)
existing_files = sorted([f for f in os.listdir(OUT_DIR) if f.startswith("fineweb_part_") and f.endswith(".txt")])
if existing_files:
    last_num = int(existing_files[-1].replace("fineweb_part_", "").replace(".txt", ""))
    docs_done = (last_num + 1) * DOCS_PER_FILE
    print(f"Found {len(existing_files)} existing files (~{docs_done:,} docs)")
    print(f"Resuming from doc {docs_done:,}...")
else:
    docs_done = 0

print(f"\nStreaming FineWeb {FINEWEB_SUBSET}, up to {MAX_DOCS:,} documents (~28 GB)...")
ds = load_dataset("HuggingFaceFW/fineweb", name=FINEWEB_SUBSET, split="train", streaming=True)

buf = []
file_idx = len(existing_files)
total_chars = 0

i = -1
for i, example in enumerate(ds):
    if i >= MAX_DOCS:
        break
    # Skip already-done docs
    if i < docs_done:
        if i % 1_000_000 == 0 and i > 0:
            print(f"  Skipping... {i:,}/{docs_done:,}")
        continue

    buf.append(example["text"])
    if len(buf) >= DOCS_PER_FILE:
        path = os.path.join(OUT_DIR, f"fineweb_part_{file_idx:04d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_chars += len(content)
        total_gb = total_chars / 1024 / 1024 / 1024
        print(f"  {path.split('/')[-1]}: {size_mb:.1f} MB | {i+1:,} docs | {total_gb:.1f} GB total")
        buf = []
        file_idx += 1

# Write remainder
if buf:
    path = os.path.join(OUT_DIR, f"fineweb_part_{file_idx:04d}.txt")
    content = SEPARATOR.join(buf)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    total_chars += len(content)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"  {path.split('/')[-1]}: {size_mb:.1f} MB ({len(buf)} docs)")
    file_idx += 1

total_gb = total_chars / 1024 / 1024 / 1024
print(f"\nDone!")
print(f"  {i + 1:,} documents in {file_idx} files")
print(f"  Total: ~{total_gb:.1f} GB")
print(f"  Saved to: {OUT_DIR}")
print(f"\nNext: tokenizer_pipeline (TRAIN_NEW_TOKENIZER=False / run_tokenizer.py with no --train) -> upload_to_drive -> pre_train")